# 07 · Run records, replay and trace

**Role of this layer.** Every run is saved as `runs/<run_id>/manifest.json` plus
one output file per step. The manifest records the request, what the LLM
proposed and what the user changed and approved, the exact data versions (file
hashes), library versions and git commit, the registry version, every step's
output hash, the narrative and its grounding check, and stage timings.

* **replay** re-runs the final plan with the recorded approvals and *no LLM*,
  after checking the data files still match, and compares every output hash.
* **trace** answers "where does this number come from?": it walks the plan
  backwards from a step to the source files.

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.config import PROJECT_ROOT
from quantlab.engine import run_request
from quantlab.lineage import load_run, replay, trace
from quantlab.scenarios import SCENARIOS_BY_ID

scenario = SCENARIOS_BY_ID["F"]
session = run_request(scenario.request, list(scenario.dataset_ids), "mock")
record = load_run(session.run_id, settings.runs_dir)
print("run:", record.run_id, "| status:", record.status)
print("manifest:", session.manifest_path.relative_to(PROJECT_ROOT))
print("registry version:", record.registry_version[:16])
print("environment:", record.environment["python"], record.environment["packages"])
pd.DataFrame([d.model_dump() for d in record.datasets])

run: 20261016-124521-97ccf6 | status: completed
manifest: runs/20261016-124521-97ccf6/manifest.json
registry version: 3ad9cf225d152d1b
environment: 3.12.14 {'pandas': '3.0.6', 'numpy': '2.5.4', 'scipy': '1.18.1', 'statsmodels': '0.15.0', 'pydantic': '2.14.1', 'openai': '3.31.0', 'anthropic': '1.14.0'}


,dataset_id,fred_id,sha256,retrieved_at,source_url,rows_used,start,end
0,DGS10,DGS10,5ec8d94ff0d1971760ded0dcd2965b2fd4d6d1086cee1df587010cb7a12c49f3,2026-10-14T19:24:42Z,https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS10,16179,1962-01-02,2026-10-09
1,DGS2,DGS2,4b4853bf93d8174d0c3b87d2d21220ddc2bc90a2e9eea2f9bca2ad0db4b0b6f4,2026-10-14T19:24:42Z,https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS2,12587,1976-06-01,2026-10-09


The saved output file of each step *is* the canonical text that was hashed, so
anyone can verify an output against the manifest with a plain SHA-256:

In [3]:
import hashlib

outputs = settings.runs_dir / record.run_id / "outputs"
for step in record.steps:
    path = next(outputs.glob(f"{step.step_id}_*"))
    same = hashlib.sha256(path.read_bytes()).hexdigest() == step.output_hash
    print(f"{path.name:28} matches manifest: {same}")

s1_load_series.csv           matches manifest: True
s2_load_series.csv           matches manifest: True
s3_resample.csv              matches manifest: True
s4_resample.csv              matches manifest: True
s5_spread.csv                matches manifest: True
s6_summary_stats.json        matches manifest: True
s7_difference.csv            matches manifest: True
s8_adf_test.json             matches manifest: True


Replay without any LLM:

In [4]:
report = replay(record.run_id, settings)
print(report.message, "| all match:", report.all_match)
pd.DataFrame([s.model_dump() for s in report.steps])

8 of 8 step outputs match; replay status completed, original status completed. | all match: True


,step_id,capability,original_hash,replay_hash,match
0,s1,load_series,29bd358698ae4591ca0a980113aabbee47a49c4d2d9250e37b8bb546c4616e8e,29bd358698ae4591ca0a980113aabbee47a49c4d2d9250e37b8bb546c4616e8e,True
1,s2,load_series,4fea097253b12777a0d6f0f090cb6325c6e521780445440b6abc25e404c37991,4fea097253b12777a0d6f0f090cb6325c6e521780445440b6abc25e404c37991,True
2,s3,resample,1a8160239c299c4558d6854287a232c14b3ccbe63f7b3cf5c5d6dfa89151c768,1a8160239c299c4558d6854287a232c14b3ccbe63f7b3cf5c5d6dfa89151c768,True
3,s4,resample,aaa7b644d23521b1dbf3805df98022a7e39575b3a4a34b136f960124645af6d7,aaa7b644d23521b1dbf3805df98022a7e39575b3a4a34b136f960124645af6d7,True
4,s5,spread,b8f592d06551f9546f12f01b822d2d11265d515bcd66fab14b28be3da881ea9f,b8f592d06551f9546f12f01b822d2d11265d515bcd66fab14b28be3da881ea9f,True
5,s6,summary_stats,d35a44da838f8e22ad21b72f1b673250e7f319026bb77c71b96282d19edb62d9,d35a44da838f8e22ad21b72f1b673250e7f319026bb77c71b96282d19edb62d9,True
6,s7,difference,585988c5c820e37c2af05ebb78df547cc0f88b3adfcc6781e195fe3d4b4fe9e3,585988c5c820e37c2af05ebb78df547cc0f88b3adfcc6781e195fe3d4b4fe9e3,True
7,s8,adf_test,3743347956cdf248c1f1977cbf7d4c2fcff097d5640741f3df8ffcb6f7f6bbdf,3743347956cdf248c1f1977cbf7d4c2fcff097d5640741f3df8ffcb6f7f6bbdf,True


Trace the ADF test (step s8) back to the FRED files:

In [5]:
lineage = trace(record, "s8")
pd.DataFrame(
    [
        {
            "step": s.step_id,
            "capability": s.capability,
            "inputs": s.inputs,
            "params": s.params,
            "output sha256": s.output_hash[:12],
        }
        for s in lineage.chain
    ]
)

,step,capability,inputs,params,output sha256
0,s8,adf_test,{'series': 's5'},{'regression': 'c'},3743347956cd
1,s5,spread,"{'a': 's3', 'b': 's4'}",{},b8f592d06551
2,s4,resample,{'series': 's2'},"{'freq': 'W', 'how': 'last'}",aaa7b644d235
3,s3,resample,{'series': 's1'},"{'freq': 'W', 'how': 'last'}",1a8160239c29
4,s2,load_series,{},{'dataset_id': 'DGS2'},4fea097253b1
5,s1,load_series,{},{'dataset_id': 'DGS10'},29bd358698ae


In [6]:
pd.DataFrame([s.model_dump() for s in lineage.sources])

,dataset_id,fred_id,sha256,retrieved_at,source_url,rows_used,start,end
0,DGS10,DGS10,5ec8d94ff0d1971760ded0dcd2965b2fd4d6d1086cee1df587010cb7a12c49f3,2026-10-14T19:24:42Z,https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS10,16179,1962-01-02,2026-10-09
1,DGS2,DGS2,4b4853bf93d8174d0c3b87d2d21220ddc2bc90a2e9eea2f9bca2ad0db4b0b6f4,2026-10-14T19:24:42Z,https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS2,12587,1976-06-01,2026-10-09
